# Prepare raw AIS data for TrAISformer

Converts a raw Marine Cadastre-style AIS CSV into the pickle format `TrAISformer`'s `datasets.AISDataset` expects.

Input CSV columns expected: `MMSI, BaseDateTime, LAT, LON, SOG, COG` (other columns like `Heading, VesselName, IMO, CallSign, VesselType, Status, Length, Width, Draft, Cargo, TransceiverClass` are ignored — the model only uses `MMSI, LAT, LON, SOG, COG, timestamp`).

Output: pickle files `<name>_train.pkl`, `<name>_valid.pkl`, `<name>_test.pkl`, each a list of dicts:
```
{"mmsi": int, "traj": np.ndarray of shape (T, 5)}   # columns: [lat_norm, lon_norm, sog_norm, cog_norm, unix_timestamp]
```

**Important — geographic ROI:** the `lat_min/lat_max/lon_min/lon_max` you choose here define the model's coordinate system (the discrete lat/lon bins it predicts over). The public TrAISformer checkpoint was trained on Danish waters (`lat 55.5-58.0`, `lon 10.3-13.0`). If your data is from a different region, these bounds must be set from *your* data, and `config_trAISformer.py`'s `lat_min/lat_max/lon_min/lon_max` (and ideally `lat_size/lon_size`) must be updated to match before you retrain/fine-tune — you can't reuse the Denmark-trained checkpoint as-is on out-of-region data.

## Config

In [1]:
import numpy as np
import pandas as pd
import pickle
import os

# ---- Set these to match your data / model config ----
INPUT_CSV = "AIS_178700220240876415_1207-1787002202965.csv"
OUTPUT_DIR = "./data/midatlantic/"
DATASET_NAME = "midatlantic"

LAT_MIN, LAT_MAX = None, None   # set explicitly to match your ROI, or leave None to auto-compute from data
LON_MIN, LON_MAX = None, None
SOG_CAP = 30.0                  # knots; repo default bin count assumes a 30-knot cap
RESAMPLE_MINUTES = 10           # matches init_seqlen/max_seqlen step assumption (10-min resolution)
MAX_GAP_HOURS = 2               # gap that splits one MMSI's pings into separate trajectories
MIN_SEQLEN = 36                 # matches config.min_seqlen
MOVING_THRESHOLD = 0.05         # normalized SOG threshold used to trim the moored/anchored start

USE_COLS = ["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG"]

## 1. Load + clean the raw CSV

In [2]:
def load_and_clean(path):
    df = pd.read_csv(path, usecols=USE_COLS)

    # parse timestamp
    df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"], errors="coerce")
    df = df.dropna(subset=["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG"])

    # drop AIS "not available" sentinel values
    df = df[(df["LAT"].between(-90, 90)) & (df["LON"].between(-180, 180))]
    df = df[df["LAT"] != 91]
    df = df[df["LON"] != 181]
    df = df[df["SOG"] < 102.3]
    df = df[(df["COG"] >= 0) & (df["COG"] < 360)]

    # drop duplicate pings, sort
    df = df.drop_duplicates(subset=["MMSI", "BaseDateTime"])
    df = df.sort_values(["MMSI", "BaseDateTime"]).reset_index(drop=True)
    return df

df = load_and_clean(INPUT_CSV)
print(f"{len(df):,} rows after cleaning, {df['MMSI'].nunique():,} unique vessels")
df.head()

17,302,744 rows after cleaning, 2,812 unique vessels


,MMSI,BaseDateTime,LAT,LON,SOG,COG
0,0,2023-01-07 22:23:55,37.12667,-76.50266,0.0,0.0
1,0,2023-01-07 22:42:17,37.12669,-76.50266,0.0,0.0
2,0,2023-01-07 22:57:33,37.12668,-76.50266,0.0,0.0
3,0,2023-01-09 21:25:46,37.12669,-76.50267,0.0,0.0
4,0,2023-01-09 21:34:58,37.12668,-76.50266,0.0,0.0


## 2. Segment into trajectories\n\nSplit each MMSI's pings into separate trajectories wherever the time gap exceeds `MAX_GAP_HOURS`, since raw AIS isn't evenly sampled.

In [3]:
def segment_trajectories(df, max_gap_hours):
    segments = []
    for mmsi, g in df.groupby("MMSI"):
        g = g.sort_values("BaseDateTime")
        gap = g["BaseDateTime"].diff().dt.total_seconds() / 3600.0
        seg_id = (gap > max_gap_hours).cumsum().fillna(0)
        for _, seg in g.groupby(seg_id):
            if len(seg) >= 2:
                segments.append((mmsi, seg))
    return segments

segments = segment_trajectories(df, MAX_GAP_HOURS)
print(f"{len(segments):,} raw trajectory segments")

18,959 raw trajectory segments


## 3. Resample to a fixed time grid\n\nInterpolate LAT/LON/SOG/COG onto a regular grid (matches the model's ~10-minute step assumption).

In [4]:
def resample_segment(seg, minutes):
    seg = seg.set_index("BaseDateTime")
    rule = f"{minutes}min"
    resampled = seg[["LAT", "LON", "SOG", "COG"]].resample(rule).mean()
    resampled = resampled.interpolate(method="linear", limit_direction="forward")
    resampled = resampled.dropna()
    resampled["timestamp"] = resampled.index.astype("int64") // 10**9
    return resampled

## 4. Normalize to [0, 1)\n\nUses the same lat/lon/sog/cog normalization the model expects.

In [5]:
def normalize(resampled, lat_min, lat_max, lon_min, lon_max, sog_cap):
    out = resampled.copy()
    out["LAT"] = ((out["LAT"] - lat_min) / (lat_max - lat_min)).clip(0, 0.9999)
    out["LON"] = ((out["LON"] - lon_min) / (lon_max - lon_min)).clip(0, 0.9999)
    out["SOG"] = (out["SOG"].clip(upper=sog_cap) / sog_cap).clip(0, 0.9999)
    out["COG"] = (out["COG"] / 360.0).clip(0, 0.9999)
    return out

def trim_to_moving(traj, moving_threshold):
    """Drop the leading moored/anchored portion (mirrors the model notebook's own logic)."""
    idx = np.where(traj[:, 2] > moving_threshold)[0]
    if len(idx) == 0:
        return traj[len(traj):]  # empty -> filtered out downstream
    return traj[idx[0]:, :]

## 5. Build trajectories + save pickle files

In [6]:
lat_min = LAT_MIN if LAT_MIN is not None else df["LAT"].min()
lat_max = LAT_MAX if LAT_MAX is not None else df["LAT"].max()
lon_min = LON_MIN if LON_MIN is not None else df["LON"].min()
lon_max = LON_MAX if LON_MAX is not None else df["LON"].max()
print(f"Using bounds: lat [{lat_min}, {lat_max}]  lon [{lon_min}, {lon_max}]")
print("NOTE: these must match config_trAISformer.py's lat_min/lat_max/lon_min/lon_max"
      " if you retrain/fine-tune the model.")

data = []
for mmsi, seg in segments:
    resampled = resample_segment(seg, RESAMPLE_MINUTES)
    if len(resampled) < MIN_SEQLEN:
        continue
    normed = normalize(resampled, lat_min, lat_max, lon_min, lon_max, SOG_CAP)
    traj = normed[["LAT", "LON", "SOG", "COG", "timestamp"]].to_numpy()
    traj = trim_to_moving(traj, MOVING_THRESHOLD)
    if len(traj) < MIN_SEQLEN or np.isnan(traj).any():
        continue
    data.append({"mmsi": int(mmsi), "traj": traj})

print(f"Built {len(data)} trajectories from {df['MMSI'].nunique()} vessels")

Using bounds: lat [36.50551, 38.69419]  lon [-77.3467, -74.19357]
NOTE: these must match config_trAISformer.py's lat_min/lat_max/lon_min/lon_max if you retrain/fine-tune the model.
Built 6495 trajectories from 2812 vessels


In [7]:
# 80/10/10 split by trajectory
rng = np.random.default_rng(42)
idx = rng.permutation(len(data))
n_train = int(0.8 * len(data))
n_valid = int(0.1 * len(data))
splits = {
    "train": [data[i] for i in idx[:n_train]],
    "valid": [data[i] for i in idx[n_train:n_train + n_valid]],
    "test": [data[i] for i in idx[n_train + n_valid:]],
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
for split, items in splits.items():
    out_path = os.path.join(OUTPUT_DIR, f"{DATASET_NAME}_{split}.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(items, f)
    print(f"  {split}: {len(items)} trajectories -> {out_path}")

  train: 5196 trajectories -> ./data/midatlantic/midatlantic_train.pkl
  valid: 649 trajectories -> ./data/midatlantic/midatlantic_valid.pkl
  test: 650 trajectories -> ./data/midatlantic/midatlantic_test.pkl


## 6. Summary statistics for the report tables

Pulls the same numbers used above into the two tables tracked across regions (Table 1 — Geographical Regions of Datasets; Table 2 — Number of Paths in Datasets), so each region's row can be copied straight in once its notebook has been run.

In [8]:
import pandas as pd

# ---- Table 1: Geographical Regions of Datasets ----
table1_row = {
    "Region": DATASET_NAME.title(),
    "Date Min": df["BaseDateTime"].min().strftime("%Y-%m-%d"),
    "Date Max": df["BaseDateTime"].max().strftime("%Y-%m-%d"),
    "Lat Min": round(lat_min, 2),
    "Lat Max": round(lat_max, 2),
    "Lon Min": round(lon_min, 2),
    "Lon Max": round(lon_max, 2),
}

# ---- Table 2: Number of Paths in Datasets ----
table2_row = {
    "Region": DATASET_NAME.title(),
    "Raw Number of Paths": len(segments),
    "Cleaned Number of Paths": len(data),
    "Training Set": len(splits["train"]),
    "Validation Set": len(splits["valid"]),
    "Testing Set": len(splits["test"]),
}

table1_df = pd.DataFrame([table1_row]).set_index("Region")
table2_df = pd.DataFrame([table2_row]).set_index("Region")

print("Table 1 row — Geographical Regions of Datasets")
display(table1_df)

print("\nTable 2 row — Number of Paths in Datasets")
display(table2_df)

assert table2_row["Cleaned Number of Paths"] == (
    table2_row["Training Set"] + table2_row["Validation Set"] + table2_row["Testing Set"]
), "Cleaned count should equal train + valid + test"

Table 1 row — Geographical Regions of Datasets


,Date Min,Date Max,Lat Min,Lat Max,Lon Min,Lon Max
Region,,,,,,
Midatlantic,2023-01-01,2023-04-01,36.51,38.69,-77.35,-74.19



Table 2 row — Number of Paths in Datasets


,Raw Number of Paths,Cleaned Number of Paths,Training Set,Validation Set,Testing Set
Region,,,,,
Midatlantic,18959,6495,5196,649,650


## Next steps

- If this is a **new region** (not the Danish CT-DMA box the public checkpoint was trained on), update `config_trAISformer.py`'s `lat_min/lat_max/lon_min/lon_max` to match the bounds printed above, then retrain/fine-tune — the existing checkpoint's bins won't transfer.
- Double check `MAX_GAP_HOURS`, `RESAMPLE_MINUTES`, and `MIN_SEQLEN` against your data's actual ping density before trusting the trajectory counts above.